# 🌿 Crop Disease Detection — EDA & Training Walkthrough

Fine-tuning **YOLOv8n** on the **PlantVillage** dataset (~20.6K leaf images, 15 disease classes — bell pepper, potato, tomato).

**Pipeline:** dataset prep → YOLO label generation → EDA → fine-tuning (Colab T4 or local GPU, ~30 min) → evaluation → Gradio demo.

> Run this on Google Colab with a T4 GPU (`Runtime → Change runtime type → T4 GPU`), or locally with CUDA.

## 1. Setup

In [ ]:
%pip -q install ultralytics kagglehub opencv-python pandas matplotlib pyyaml tqdm tabulate

import torch
print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# On Colab: clone the repo so prepare_dataset.py / configs are available.
# Locally: skip this cell and run the notebook from the repo's notebooks/ dir.
import os
if not os.path.exists('prepare_dataset.py'):
    if os.path.exists('../prepare_dataset.py'):
        %cd ..
    else:
        !git clone https://github.com/YOUR_USERNAME/crop-disease-detector.git
        %cd crop-disease-detector
!ls

## 2. Dataset: convert to YOLO format

PlantVillage is a *classification* dataset — one leaf per image, no boxes. `prepare_dataset.py`:
1. reads a manual download at `data/archive/PlantVillage/` (or fetches from Kaggle with `--download`, which needs `~/.kaggle/kaggle.json` — upload yours below on Colab),
2. segments the leaf from the plain background with OpenCV (Otsu threshold on saturation) to derive one bounding box per image,
3. writes a stratified **70/20/10 train/val/test split** in YOLO layout.

In [ ]:
# Colab only, and only if using --download: upload kaggle.json
# (Kaggle → Account → Create New API Token). Skip locally — the script
# reads your manual download at data/archive/PlantVillage/ by default.
import os
NEED_KAGGLE = False  # set True if you'll run prepare_dataset.py --download
if NEED_KAGGLE and not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
    from google.colab import files
    files.upload()  # pick kaggle.json
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    os.replace('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

In [ ]:
!python prepare_dataset.py   # add --download if no manual copy; --limit 100 for a smoke run

## 3. EDA — class balance and sample images

In [ ]:
from pathlib import Path
from collections import Counter
import pandas as pd
import yaml
import matplotlib.pyplot as plt

names = yaml.safe_load(Path('data/data.yaml').read_text(encoding='utf-8'))['names']

counts = Counter()
for lbl in Path('data/plantvillage_yolo/labels/train').glob('*.txt'):
    class_id = int(lbl.read_text().split()[0])
    counts[names[class_id]] += 1

dist = pd.Series(counts).sort_values()
ax = dist.plot.barh(figsize=(9, 11), color='seagreen')
ax.set_title(f'Training images per class ({dist.sum():,} total, {len(dist)} classes)')
ax.set_xlabel('images')
plt.tight_layout()
plt.show()

print(f"Imbalance ratio (max/min): {dist.max() / dist.min():.1f}x")

In [ ]:
# Sample images with their auto-generated bounding boxes
import cv2
import random

img_dir = Path('data/plantvillage_yolo/images/train')
lbl_dir = Path('data/plantvillage_yolo/labels/train')
picks = random.Random(7).sample(sorted(img_dir.glob('*.*')), 8)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, p in zip(axes.flat, picks):
    img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    cid, cx, cy, bw, bh = map(float, (lbl_dir / f'{p.stem}.txt').read_text().split())
    x0, y0 = int((cx - bw/2) * w), int((cy - bh/2) * h)
    x1, y1 = int((cx + bw/2) * w), int((cy + bh/2) * h)
    cv2.rectangle(img, (x0, y0), (x1, y1), (255, 60, 60), 3)
    ax.imshow(img)
    ax.set_title(names[int(cid)], fontsize=8)
    ax.axis('off')
plt.suptitle('Auto-generated leaf bounding boxes (OpenCV segmentation)')
plt.tight_layout()
plt.show()

## 4. Fine-tune YOLOv8n

Starts from COCO-pretrained `yolov8n.pt`. Hyperparameters live in `configs/yolov8_finetune.yaml` — 50 epochs, early stopping at 10 stale epochs, moderate augmentation (heavy mosaic hurts studio-style single-leaf images).

~30 min on a Colab T4 for the full dataset.

In [ ]:
!python train.py --device 0

## 5. Evaluate — per-class metrics, curves, confusion matrix, annotated samples

In [ ]:
!python evaluate.py

In [ ]:
from IPython.display import Image as IPImage, display
import pandas as pd

display(IPImage('results/training_curves.png'))
display(IPImage('results/confusion_matrix.png', width=900))

df = pd.read_csv('results/per_class_metrics.csv')
print('Weakest 10 classes by mAP50:')
display(df.head(10))
print('Strongest 5:')
display(df.tail(5))

In [ ]:
# Annotated sample detections
from pathlib import Path
samples = sorted(Path('results/sample_detections').glob('*.*'))[:6]
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, p in zip(axes.flat, samples):
    ax.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
    ax.axis('off')
plt.suptitle('Test-set detections with confidence scores')
plt.tight_layout()
plt.show()

## 6. Quick inference sanity check

In [ ]:
from ultralytics import YOLO
from treatments import treatment_for

model = YOLO('runs/detect/plantvillage_finetune/weights/best.pt')
test_img = next(Path('data/plantvillage_yolo/images/test').glob('*.*'))
result = model.predict(str(test_img), conf=0.4, verbose=False)[0]

for box in result.boxes:
    name = model.names[int(box.cls)]
    print(f'{name}  ({float(box.conf):.0%})')
    print(f'  → {treatment_for(name)}')

plt.figure(figsize=(6, 6))
plt.imshow(result.plot()[:, :, ::-1])
plt.axis('off')
plt.show()

## 7. Ship it

1. **Download weights** (Colab): `runs/detect/plantvillage_finetune/weights/best.pt`
2. **Local demo**: `python app.py`
3. **Hugging Face Spaces**: create a Gradio Space, push `app.py`, `treatments.py`, `requirements.txt`, and the weights as `weights/best.pt` — the Space serves the public demo URL.

**Honest limitations:** PlantVillage is studio-style single leaves on plain backgrounds; field photos with clutter will underperform. Boxes were auto-derived from segmentation, so localization quality reflects that — this is detection-style packaging of a classification dataset, stated openly.